# Sync Author ORCID Curations

Syncs ORCID curations from the users Heroku Postgres database to a local Databricks Delta table. Runs in `jobs/authors.yaml` upstream of `ApplyAuthorOrcidCurations` and `CreateAuthors`. Mirrors `SyncAuthorNameCurations`.

**Source** (typed views over `openalex_users.public.curations`):
- `author_orcid_curations` — `replace`: set the correct ORCID for an author profile (`new_orcid`)
- `author_orcid_removals` — `remove`: detach a wrong ORCID from an author profile (`removed_orcid`)

Both values are guaranteed full-URL form `https://orcid.org/0000-...` (producer validates ISO-7064 checksum + canonicalizes at intake), the same format stored in `openalex.authors.authors.orcid`.

**Target**:
- `openalex.authors.author_orcid_curations` — one row per curated author, `(author_id, curated_orcid, removed_orcid, updated_datetime)`. An author may carry both a `replace` (set X) and a `remove` (detach Y) signal — kept as separate columns via a FULL OUTER JOIN.
- `openalex.authors.author_orcid_removals_all` (oxjob #1715 gap E): one row per `(author_id, removed_orcid)`, **every** live removal, not only the latest. `CreateAuthors` and `ApplyAuthorOrcidCurations` test membership here, so a second removal on a profile no longer brings the first ORCID back. A removal drops out only the way the latest-wins MERGE cancels one: the author's latest `replace` sets that same ORCID and is newer than the removal.

When multiple curators act on the same author, latest-wins by source `created` timestamp.

The curated ORCID is applied as an **override** in `CreateAuthors` (COALESCE / detach), not written in-place to `openalex.authors.authors`. Deleting a curation from the Heroku source causes the MERGE's `WHEN NOT MATCHED BY SOURCE THEN DELETE` to remove it here, and the next `CreateAuthors` run falls back to the organic ORCID.

## Create target table (idempotent)

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS openalex.authors.author_orcid_curations (
    author_id         BIGINT NOT NULL,
    curated_orcid     STRING,
    removed_orcid     STRING,
    updated_datetime  TIMESTAMP
)
USING DELTA
CLUSTER BY (author_id)

## Preview the latest-wins curated/removed ORCID per author

In [ ]:
%sql
WITH latest_replace AS (
  SELECT author_id, new_orcid AS curated_orcid, created AS replace_created
  FROM (
    SELECT author_id, new_orcid, created,
           ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY created DESC) AS rn
    FROM openalex_users.public.author_orcid_curations
  ) WHERE rn = 1
),
latest_remove AS (
  SELECT author_id, removed_orcid, created AS remove_created
  FROM (
    SELECT author_id, removed_orcid, created,
           ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY created DESC) AS rn
    FROM openalex_users.public.author_orcid_removals
  ) WHERE rn = 1
)
SELECT COALESCE(r.author_id, d.author_id) AS author_id,
       -- Latest wins ACROSS directions (#1442, 2026-09-30): a remove of the same ORCID newer than the
       -- replace cancels the replace; a replace newer than a remove of the same ORCID cancels the remove.
       -- Without this, "set X on 9/1, remove X on 9/9" kept serving X (CreateAuthors ranks curated_orcid first).
       CASE WHEN d.removed_orcid = r.curated_orcid AND d.remove_created > r.replace_created
            THEN NULL ELSE r.curated_orcid END AS curated_orcid,
       CASE WHEN d.removed_orcid = r.curated_orcid AND r.replace_created > d.remove_created
            THEN NULL ELSE d.removed_orcid END AS removed_orcid
FROM latest_replace r
FULL OUTER JOIN latest_remove d ON r.author_id = d.author_id

## Sync ORCID curations

Latest-wins per author for each direction, FULL OUTER JOINed so an author can hold both a `replace` and a `remove`. `WHEN NOT MATCHED BY SOURCE THEN DELETE` reverts authors whose curation was deleted upstream.

In [ ]:
%sql
MERGE INTO openalex.authors.author_orcid_curations AS target
USING (
  WITH latest_replace AS (
    SELECT author_id, new_orcid AS curated_orcid, created AS replace_created
    FROM (
      SELECT author_id, new_orcid, created,
             ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY created DESC) AS rn
      FROM openalex_users.public.author_orcid_curations
    ) WHERE rn = 1
  ),
  latest_remove AS (
    SELECT author_id, removed_orcid, created AS remove_created
    FROM (
      SELECT author_id, removed_orcid, created,
             ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY created DESC) AS rn
      FROM openalex_users.public.author_orcid_removals
    ) WHERE rn = 1
  )
  SELECT COALESCE(r.author_id, d.author_id) AS author_id,
         -- Latest wins ACROSS directions (#1442, 2026-09-30): a remove of the same ORCID newer than the
         -- replace cancels the replace; a replace newer than a remove of the same ORCID cancels the remove.
         -- Without this, "set X on 9/1, remove X on 9/9" kept serving X (CreateAuthors ranks curated_orcid first).
         CASE WHEN d.removed_orcid = r.curated_orcid AND d.remove_created > r.replace_created
              THEN NULL ELSE r.curated_orcid END AS curated_orcid,
         CASE WHEN d.removed_orcid = r.curated_orcid AND r.replace_created > d.remove_created
              THEN NULL ELSE d.removed_orcid END AS removed_orcid,
         CURRENT_TIMESTAMP() AS updated_datetime
  FROM latest_replace r
  FULL OUTER JOIN latest_remove d ON r.author_id = d.author_id
) AS source
ON target.author_id = source.author_id
WHEN MATCHED THEN UPDATE SET
  target.curated_orcid    = source.curated_orcid,
  target.removed_orcid    = source.removed_orcid,
  target.updated_datetime = source.updated_datetime
WHEN NOT MATCHED THEN INSERT (author_id, curated_orcid, removed_orcid, updated_datetime)
  VALUES (source.author_id, source.curated_orcid, source.removed_orcid, source.updated_datetime)
WHEN NOT MATCHED BY SOURCE THEN DELETE

In [ ]:
%sql
-- oxjob #1715 gap E: every live ORCID removal per author. author_orcid_curations keeps one removed_orcid per author
-- (latest wins), so a second removal on a profile undid the first: the first ORCID came back as soon as a trusted seat
-- carried it (A5101761266: removing 0000-0002-3646-2421 would bring back 0000-0002-9197-0736, removed 11 Nov 2026).
CREATE TABLE IF NOT EXISTS openalex.authors.author_orcid_removals_all (
    author_id         BIGINT NOT NULL,
    removed_orcid     STRING NOT NULL,
    updated_datetime  TIMESTAMP
)
USING DELTA
CLUSTER BY (author_id)

In [ ]:
%sql
-- oxjob #1715 gap E: all removals, not only the latest. A removal is cancelled exactly as the MERGE above cancels one:
-- the author's latest replace sets that same ORCID and is newer than the removal's latest copy. Measured 2026-11-22:
-- 87 removals on 87 authors, none with two, so on that day this table equals author_orcid_curations.removed_orcid
-- (0 profiles change). Revert: drop this cell and the one above, and the joins in CreateAuthors and
-- ApplyAuthorOrcidCurations.
MERGE INTO openalex.authors.author_orcid_removals_all AS target
USING (
  WITH latest_replace AS (
    SELECT author_id, new_orcid AS curated_orcid, created AS replace_created
    FROM (
      SELECT author_id, new_orcid, created,
             ROW_NUMBER() OVER (PARTITION BY author_id ORDER BY created DESC) AS rn
      FROM openalex_users.public.author_orcid_curations
    ) WHERE rn = 1
  ),
  removals AS (
    SELECT author_id, removed_orcid, MAX(created) AS remove_created
    FROM openalex_users.public.author_orcid_removals
    WHERE removed_orcid IS NOT NULL AND removed_orcid <> ''
    GROUP BY author_id, removed_orcid
  )
  SELECT d.author_id, d.removed_orcid
  FROM removals d
  LEFT JOIN latest_replace r ON r.author_id = d.author_id
  WHERE NOT COALESCE(d.removed_orcid = r.curated_orcid AND r.replace_created > d.remove_created, FALSE)
) AS source
ON target.author_id = source.author_id AND target.removed_orcid = source.removed_orcid
WHEN NOT MATCHED THEN INSERT (author_id, removed_orcid, updated_datetime)
  VALUES (source.author_id, source.removed_orcid, CURRENT_TIMESTAMP())
WHEN NOT MATCHED BY SOURCE THEN DELETE

## Collision guard (out-of-scope merge case)

A `replace` that sets an ORCID **already held by a different author** would leave two `authors` rows sharing one ORCID, making the global ORCID match key ambiguous on the next clustering cycle. Per the #410 design that situation is a **merge**, not an ORCID curation (out of scope here). This cell surfaces any such collisions so they are visible rather than silently created — a non-empty result is a flag to investigate, not a hard failure.

In [ ]:
%sql
SELECT c.author_id      AS curated_author_id,
       c.curated_orcid,
       a.id             AS existing_author_id
FROM openalex.authors.author_orcid_curations c
JOIN openalex.authors.authors a
  ON a.orcid = c.curated_orcid
 AND a.id <> c.author_id
WHERE c.curated_orcid IS NOT NULL AND c.curated_orcid <> ''

## Verify sync results

In [ ]:
%sql
SELECT
  COUNT(*)              AS total_curated_authors,
  COUNT(curated_orcid)  AS total_replace_signals,
  COUNT(removed_orcid)  AS total_remove_signals,
  MAX(updated_datetime) AS last_sync
FROM openalex.authors.author_orcid_curations

In [ ]:
%sql
SELECT * FROM openalex.authors.author_orcid_curations
ORDER BY updated_datetime DESC
LIMIT 100